# 58 — LGBM with BindingDB Nuclear Receptor Data

LGBM incorporating BindingDB binding data for nuclear receptors.
BindingDB provides Ki/Kd/IC50 measurements with more binding-mode diversity than ChEMBL.

Key steps:
1. Load `bindingdb_nr_data.parquet` (or fetch via BindingDB REST API).
2. Source analysis: Tanimoto proximity to test set and cliff pairs.
3. LGBM with BindingDB PXR (w=0.6) and other NR (w=0.3).
4. Save OOF + submission.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import math
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, standardize_smiles, to_inchikey, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1
)

tr = load_train()
te = load_test()
print(f'Train: {len(tr):,}  Test: {len(te):,}')
train_inchikeys = set(tr['smiles'].map(to_inchikey).dropna())

Train: 4,139  Test: 513


## 1. Load BindingDB NR data

In [2]:
BINDINGDB_CACHE = DATA_EXTERNAL / 'bindingdb_nr_data.parquet'

# Approximate pEC50 corrections for different binding measurement types
AFFINITY_OFFSETS = {'IC50': 0.0, 'EC50': 0.0, 'KI': -0.3, 'KD': -0.5}
IC50_MIN_NM = 0.01
IC50_MAX_NM = 100_000


def affinity_nm_to_pec50(val_nm: float, affinity_type: str) -> float | None:
    """Convert nM binding value to pEC50-equivalent."""
    if val_nm <= 0 or not math.isfinite(val_nm):
        return None
    if not (IC50_MIN_NM <= val_nm <= IC50_MAX_NM):
        return None
    offset = AFFINITY_OFFSETS.get(str(affinity_type).upper().strip(), 0.0)
    return -math.log10(val_nm * 1e-9) + offset


if BINDINGDB_CACHE.exists():
    print(f'Loading cached BindingDB data from {BINDINGDB_CACHE}')
    bdb_df = pd.read_parquet(BINDINGDB_CACHE)
    print(f'  {len(bdb_df):,} rows loaded')
    if 'target_name' in bdb_df.columns:
        print(bdb_df.groupby('target_name').size().to_string())
else:
    print('bindingdb_nr_data.parquet not found — attempting REST API fetch for PXR...')
    import requests, time

    NR_UNIPROTS = {'PXR': 'O75469', 'VDR': 'P11473', 'FXR': 'Q96RI1', 'PPARg': 'P37231'}
    BDB_URL = ('https://bindingdb.org/axis2/services/BDBService/getLigandsByUniprots'
               '?uniprot={uid}&response=json')

    all_rows = []
    for target_name, uniprot in NR_UNIPROTS.items():
        url = BDB_URL.format(uid=uniprot)
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            records = data.get('affinities', [])
            if isinstance(records, dict):
                records = records.get('affinity', [])
            if not isinstance(records, list):
                records = [records]
            for rec in records:
                if not isinstance(rec, dict):
                    continue
                smi = rec.get('smiles', '') or rec.get('ligand_smiles', '')
                if not smi:
                    continue
                atype = str(rec.get('affinity_type', '') or '').upper()
                if atype not in ('IC50', 'EC50', 'KI', 'KD'):
                    continue
                try:
                    val_nm = float(rec.get('affinity', 'nan'))
                except (ValueError, TypeError):
                    continue
                pval = affinity_nm_to_pec50(val_nm, atype)
                if pval is not None and 3.0 <= pval <= 11.0:
                    all_rows.append({
                        'smiles': smi, 'pec50': pval,
                        'affinity_nm': val_nm, 'affinity_type': atype,
                        'target_name': target_name, 'uniprot': uniprot,
                    })
            print(f'  {target_name}: retrieved {len(records):,} records')
        except Exception as e:
            print(f'  {target_name}: fetch failed — {e}')
        time.sleep(0.5)

    if all_rows:
        bdb_df = pd.DataFrame(all_rows)
        bdb_df['std_smiles'] = bdb_df['smiles'].map(standardize_smiles)
        bdb_df['inchikey'] = bdb_df['std_smiles'].map(lambda s: to_inchikey(s) if s else None)
        bdb_df = bdb_df.dropna(subset=['std_smiles', 'inchikey']).reset_index(drop=True)
        bdb_df = (
            bdb_df.sort_values('pec50', ascending=False)
                  .drop_duplicates(subset=['inchikey', 'target_name'], keep='first')
                  .reset_index(drop=True)
        )
        bdb_df.to_parquet(BINDINGDB_CACHE, index=False)
        print(f'Saved {len(bdb_df):,} records to {BINDINGDB_CACHE}')
    else:
        print('No BindingDB data retrieved — proceeding with CRC-only baseline')
        bdb_df = pd.DataFrame(
            columns=['smiles', 'std_smiles', 'inchikey', 'pec50',
                     'affinity_type', 'target_name', 'uniprot'])
        bdb_df.to_parquet(BINDINGDB_CACHE, index=False)

# Ensure required columns and filter
pec50_col = 'pec50' if 'pec50' in bdb_df.columns else 'pchembl_value'
if pec50_col in bdb_df.columns:
    bdb_df = bdb_df.rename(columns={pec50_col: 'pec50'}) if pec50_col != 'pec50' else bdb_df
    bdb_df = bdb_df[(bdb_df['pec50'] >= 3.0) & (bdb_df['pec50'] <= 11.0)]

# Remove PXR training overlap
if 'inchikey' not in bdb_df.columns:
    smi_col = next((c for c in ['std_smiles', 'smiles'] if c in bdb_df.columns), None)
    if smi_col:
        bdb_df['inchikey'] = bdb_df[smi_col].map(to_inchikey)
bdb_df = bdb_df.dropna(subset=['inchikey'])
bdb_df = bdb_df[~bdb_df['inchikey'].isin(train_inchikeys)]
print(f'\nAfter removing PXR train overlap: {len(bdb_df):,} BindingDB records')

Loading cached BindingDB data from D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\external\bindingdb_nr_data.parquet
  5,690 rows loaded
target_name
FXR       456
LXRa      856
PPARg    2493
PXR       346
RXRa     1061
VDR       478

After removing PXR train overlap: 5,572 BindingDB records


## 2. Source analysis — proximity to test set and cliff pairs

In [3]:
if len(bdb_df) > 0:
    smi_col = next((c for c in ['std_smiles', 'smiles'] if c in bdb_df.columns), None)
    bdb_df['smiles_use'] = bdb_df[smi_col]

    print('Computing Tanimoto proximity of BindingDB compounds to test set...')
    fps_bdb = morgan_fp_batch(bdb_df['smiles_use'].tolist()).astype(np.float32)
    fps_te  = morgan_fp_batch(te['smiles'].tolist()).astype(np.float32)

    # Batch Tanimoto: max similarity per BindingDB compound to any test compound
    BATCH = 512
    max_sim_to_test = np.zeros(len(bdb_df), dtype=np.float32)
    for i in range(0, len(fps_bdb), BATCH):
        chunk = fps_bdb[i:i+BATCH]
        dot = chunk @ fps_te.T
        rs_chunk = chunk.sum(1, keepdims=True)
        rs_te = fps_te.sum(1)[None, :]
        union = rs_chunk + rs_te - dot
        with np.errstate(divide='ignore', invalid='ignore'):
            tan = np.where(union > 0, dot / union, 0.0)
        max_sim_to_test[i:i+BATCH] = tan.max(axis=1)

    n_close_to_test = (max_sim_to_test >= 0.5).sum()
    print(f'  BindingDB compounds with Tanimoto >= 0.5 to any test compound: '
          f'{n_close_to_test:,} ({n_close_to_test/len(bdb_df)*100:.1f}%)')
    print(f'  Median max Tanimoto to test: {np.median(max_sim_to_test):.3f}')
    print(f'  Mean max Tanimoto to test:   {max_sim_to_test.mean():.3f}')

    # Proximity to cliff pairs
    cliff_path = DATA_PROCESSED / 'cliff_labels.parquet'
    if cliff_path.exists():
        cliff_df = pd.read_parquet(cliff_path)
        cliff_smiles_col = 'smiles' if 'smiles' in cliff_df.columns else None
        if cliff_smiles_col:
            cliff_mask_tr = tr['smiles'].isin(
                cliff_df.loc[cliff_df.get('cliff_role', pd.Series(dtype=int)).fillna(0).ne(0), cliff_smiles_col]
            ) if 'cliff_role' in cliff_df.columns else pd.Series([False]*len(tr))
            n_cliff = cliff_mask_tr.sum()
            if n_cliff > 0:
                fps_cliff = morgan_fp_batch(
                    tr.loc[cliff_mask_tr, 'smiles'].tolist()
                ).astype(np.float32)
                dot_c = fps_bdb @ fps_cliff.T
                rs_c = fps_cliff.sum(1)[None, :]
                rs_b = fps_bdb.sum(1)[:, None]
                union_c = rs_b + rs_c - dot_c
                with np.errstate(divide='ignore', invalid='ignore'):
                    tan_c = np.where(union_c > 0, dot_c / union_c, 0.0)
                n_close_cliff = (tan_c.max(axis=1) >= 0.5).sum()
                print(f'  BindingDB compounds close (>=0.5) to cliff pairs: '
                      f'{n_close_cliff:,} ({n_close_cliff/len(bdb_df)*100:.1f}%)')
    else:
        print('  cliff_labels.parquet not found — skipping cliff proximity')

    # Target-wise statistics
    if 'target_name' in bdb_df.columns:
        print('\nPer-target pEC50 stats:')
        print(bdb_df.groupby('target_name')['pec50'].describe().round(2).to_string())
else:
    print('No BindingDB data — skipping proximity analysis.')

Computing Tanimoto proximity of BindingDB compounds to test set...


  BindingDB compounds with Tanimoto >= 0.5 to any test compound: 0 (0.0%)
  Median max Tanimoto to test: 0.241
  Mean max Tanimoto to test:   0.242
  BindingDB compounds close (>=0.5) to cliff pairs: 13 (0.2%)

Per-target pEC50 stats:
              count  mean   std   min   25%   50%   75%    max
target_name                                                   
FXR           451.0  5.96  1.38  4.00  5.00  5.49  6.70  10.30
LXRa          840.0  6.56  1.07  4.04  5.77  6.60  7.24   9.10
PPARg        2401.0  6.64  1.19  4.00  5.68  6.57  7.57  10.30
PXR           342.0  6.20  0.89  4.39  5.60  6.12  6.67   9.00
RXRa         1061.0  6.97  1.04  4.08  6.14  6.99  7.82   9.40
VDR           477.0  6.45  1.60  4.08  4.98  6.16  8.04  10.54


## 3. LGBM with BindingDB

In [4]:
# ── Featurize ────────────────────────────────────────────────────────────────
print('Featurizing PXR training set...')
X_tr = impute(combined(tr['smiles'].tolist()))
y_tr = tr['pec50'].values.astype(np.float32)
scaffolds_tr = tr['smiles'].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds_tr, n_splits=N_FOLDS, seed=SEED)

X_te = impute(combined(te['smiles'].tolist()))
print(f'X_tr: {X_tr.shape}  X_te: {X_te.shape}')

X_bdb = None
y_bdb = None
w_bdb = None

if len(bdb_df) > 0 and 'pec50' in bdb_df.columns:
    print(f'Featurizing {len(bdb_df):,} BindingDB compounds...')
    bdb_df = bdb_df.dropna(subset=['smiles_use', 'pec50'])
    X_bdb = impute(combined(bdb_df['smiles_use'].tolist()))
    y_bdb = bdb_df['pec50'].values.astype(np.float32)

    # Per-target weights: PXR=0.6, others=0.3
    if 'target_name' in bdb_df.columns:
        w_bdb = bdb_df['target_name'].map(lambda t: 0.6 if t == 'PXR' else 0.3)
        w_bdb = w_bdb.values.astype(np.float32)
    else:
        w_bdb = np.full(len(bdb_df), 0.4, dtype=np.float32)

    print(f'X_bdb: {X_bdb.shape}')
    print(f'Weight distribution: PXR w=0.6: {(w_bdb == 0.6).sum():,}  '
          f'Other NR w=0.3: {(w_bdb == 0.3).sum():,}')

# ── Scaffold CV ──────────────────────────────────────────────────────────────
oof_bdb = np.full(len(y_tr), np.nan, dtype=np.float32)
oof_crc = np.full(len(y_tr), np.nan, dtype=np.float32)
w_crc = np.ones(len(y_tr), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # CRC-only baseline
    m_crc = lgb.LGBMRegressor(**LGBM_PARAMS)
    m_crc.fit(X_tr[tr_idx], y_tr[tr_idx],
              callbacks=[lgb.log_evaluation(-1)])
    oof_crc[va_idx] = m_crc.predict(X_tr[va_idx])

    # With BindingDB
    if X_bdb is not None:
        X_aug = np.vstack([X_tr[tr_idx], X_bdb])
        y_aug = np.concatenate([y_tr[tr_idx], y_bdb])
        w_aug = np.concatenate([w_crc[tr_idx], w_bdb])
    else:
        X_aug, y_aug, w_aug = X_tr[tr_idx], y_tr[tr_idx], w_crc[tr_idx]

    m_bdb = lgb.LGBMRegressor(**LGBM_PARAMS)
    m_bdb.fit(X_aug, y_aug, sample_weight=w_aug, callbacks=[lgb.log_evaluation(-1)])
    oof_bdb[va_idx] = m_bdb.predict(X_tr[va_idx])

    fold_rae_crc = rae(y_tr[va_idx], oof_crc[va_idx])
    fold_rae_bdb = rae(y_tr[va_idx], oof_bdb[va_idx])
    print(f'  Fold {fold+1}: CRC RAE={fold_rae_crc:.4f}  +BindingDB RAE={fold_rae_bdb:.4f}')

rae_crc = rae(y_tr, oof_crc)
rae_bdb = rae(y_tr, oof_bdb)
print(f'\nOOF RAE (CRC only):      {rae_crc:.4f}')
print(f'OOF RAE (+BindingDB):    {rae_bdb:.4f}')
print(f'Delta:                   {rae_bdb - rae_crc:+.4f}')

Featurizing PXR training set...


X_tr: (4139, 2265)  X_te: (513, 2265)
Featurizing 5,572 BindingDB compounds...


X_bdb: (5572, 2265)
Weight distribution: PXR w=0.6: 342  Other NR w=0.3: 5,230


  Fold 1: CRC RAE=0.4934  +BindingDB RAE=0.5005


  Fold 2: CRC RAE=0.5762  +BindingDB RAE=0.6197


  Fold 3: CRC RAE=0.5961  +BindingDB RAE=0.6159


  Fold 4: CRC RAE=0.5635  +BindingDB RAE=0.5868


  Fold 5: CRC RAE=0.5952  +BindingDB RAE=0.6237

OOF RAE (CRC only):      0.5600
OOF RAE (+BindingDB):    0.5839
Delta:                   +0.0239


## 4. Save

In [5]:
# Use best OOF
best_oof = oof_bdb if rae_bdb <= rae_crc else oof_crc
best_label = '+BindingDB' if rae_bdb <= rae_crc else 'CRC only'
print(f'Using: {best_label}  OOF RAE = {rae(y_tr, best_oof):.4f}')

# Final model on all data
if X_bdb is not None and rae_bdb <= rae_crc:
    X_final = np.vstack([X_tr, X_bdb])
    y_final = np.concatenate([y_tr, y_bdb])
    w_final = np.concatenate([np.ones(len(y_tr), dtype=np.float32), w_bdb])
else:
    X_final, y_final, w_final = X_tr, y_tr, np.ones(len(y_tr), dtype=np.float32)

final_model = lgb.LGBMRegressor(**LGBM_PARAMS)
final_model.fit(X_final, y_final, sample_weight=w_final, callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(final_model.predict(X_te),
                   float(y_tr.min()) - 0.5, float(y_tr.max()) + 0.5)

np.save(DATA_PROCESSED / 'oof_lgbm_bindingdb.npy', best_oof)
np.save(DATA_PROCESSED / 'te_lgbm_bindingdb.npy', te_preds)
print(f'Saved oof_lgbm_bindingdb.npy  OOF RAE = {rae(y_tr, best_oof):.4f}')

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out_path = SUBMISSIONS / '58_lgbm_bindingdb.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'\nTest preds: mean={te_preds.mean():.3f}  std={te_preds.std():.3f}')
sub['pEC50'].describe().round(3)

Using: CRC only  OOF RAE = 0.5600


Saved oof_lgbm_bindingdb.npy  OOF RAE = 0.5600
Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\58_lgbm_bindingdb.csv

Test preds: mean=4.804  std=0.649


count    513.000
mean       4.804
std        0.650
min        2.209
25%        4.443
50%        4.948
75%        5.282
max        6.052
Name: pEC50, dtype: float64